# W02 Agentic Evaluation — ReAct Agent Loop

**InGen AI Model Evaluation · Week 2**

This notebook loads `W02_Agentic_Eval_results.json` and presents the multi-step agentic evaluation findings across 20 TrackB scenarios × 2 providers (Anthropic claude-sonnet-4-6, DeepSeek deepseek-chat).

**Three SREGym metrics (numeric, per scenario):**
1. **task_completion_rate** — 1 if all required steps verified successful and no early-exit failure; else 0
2. **step_efficiency** — `actual_actions_taken / n_required_steps` (≥ 1.0; closer to 1.0 is better)
3. **error_recovery_rate** — of error observations, fraction where the agent took a corrective next action

Step verification uses 3 judge seeds (42/100/2026) per step for Krippendorff's alpha.

In [ ]:
import json
from pathlib import Path
import pandas as pd

REPO_ROOT = Path().resolve().parents[0]
RESULTS_PATH = REPO_ROOT / "week02_evaluation" / "W02_Agentic_Eval_results.json"

with RESULTS_PATH.open() as f:
    data = json.load(f)

meta = data["metadata"]
df = pd.DataFrame(data["results"])

print(f"Total results:  {len(df)}")
print(f"Providers:      {df['provider'].unique().tolist()}")
print(f"Platforms:      {df['platform'].unique().tolist()}")
print(f"Dry-run:        {meta.get('dry_run', False)}")
print(f"Errors:         {df['error'].notna().sum()}")

## Data Quality Check

> **Assertion**: `step_efficiency` must be populated (non-null) for all 40 rows before summary statistics are reported.

In [ ]:
expected_rows = 20 * 2  # 20 scenarios × 2 providers

missing_eff = df[df["step_efficiency"].isna()]
assert len(missing_eff) == 0, (
    f"step_efficiency is NULL for {len(missing_eff)} rows: "
    + missing_eff[["scenario_id", "provider"]].to_string()
)

eff_below_1 = df[df["step_efficiency"] < 1.0]
if len(eff_below_1) > 0:
    print(f"⚠  WARNING: {len(eff_below_1)} rows have step_efficiency < 1.0 "
          "(indicates transcript parse issue, not a valid agent result):")
    print(eff_below_1[["scenario_id", "provider", "step_efficiency", "actual_actions_taken",
                        "n_required_steps"]].to_string())
else:
    print(f"✓ All {len(df)} rows have step_efficiency populated and ≥ 1.0")

if len(df) != expected_rows and not meta.get('dry_run'):
    print(f"⚠  Expected {expected_rows} rows, got {len(df)}")
else:
    print(f"✓ Row count: {len(df)} (expected {expected_rows})")

## Metric Summary: Provider Comparison

In [ ]:
metrics = ["task_completion_rate", "step_efficiency", "error_recovery_rate"]

summary = (
    df.groupby("provider")[metrics]
    .agg(["mean", "median"])
    .round(3)
)
summary.columns = [f"{m}_{stat}" for m, stat in summary.columns]
summary = summary.reset_index()
print("Provider-level summary:")
summary

## Metric Summary: Per Platform

In [ ]:
platform_summary = (
    df.groupby(["platform", "provider"])[metrics]
    .mean()
    .round(3)
    .reset_index()
)
platform_summary

## Per-Scenario Step Verdicts

In [ ]:
scenario_df = df[["scenario_id", "platform", "provider",
                  "task_completion_rate", "step_efficiency",
                  "error_recovery_rate", "early_exit_triggered",
                  "actual_actions_taken", "n_required_steps"]].copy()
scenario_df = scenario_df.sort_values(["platform", "scenario_id", "provider"])
scenario_df

## Task Completion Rate by Platform and Provider

In [ ]:
try:
    import plotly.graph_objects as go

    pivot = df.pivot_table(
        index="platform", columns="provider",
        values="task_completion_rate", aggfunc="mean"
    ).round(2).reset_index()

    colors = {"anthropic": "#C97B44", "deepseek": "#4A90D9"}
    fig = go.Figure()
    for provider in ["anthropic", "deepseek"]:
        if provider in pivot.columns:
            fig.add_trace(go.Bar(
                name=provider,
                x=pivot["platform"],
                y=pivot[provider],
                marker_color=colors.get(provider, "#888"),
                text=pivot[provider].map(lambda v: f"{v:.2f}"),
                textposition="outside",
            ))

    fig.update_layout(
        title="Task Completion Rate by Platform and Provider",
        yaxis=dict(range=[0, 1.15], title="Task Completion Rate"),
        barmode="group",
        template="plotly_white",
        legend=dict(orientation="h", y=-0.2),
        width=800, height=420,
    )
    fig.show()
except ImportError:
    print("plotly not installed — skipping chart")

## Step Efficiency Distribution

In [ ]:
try:
    import plotly.graph_objects as go

    fig = go.Figure()
    for provider in df["provider"].unique():
        fig.add_trace(go.Box(
            y=df[df["provider"] == provider]["step_efficiency"],
            name=provider,
            boxmean=True,
        ))

    fig.update_layout(
        title="Step Efficiency Distribution (lower is better; 1.0 = optimal)",
        yaxis_title="step_efficiency (actions / required_steps)",
        template="plotly_white",
        width=600, height=400,
    )
    fig.show()
except ImportError:
    print("plotly not installed — skipping chart")

## Transcript Sample — First Successful Run

Shows the Thought/Action/Observation sequence for the first completed scenario.

In [ ]:
completed = df[df["task_completion_rate"] == 1]
if len(completed) > 0:
    sample = completed.iloc[0]
    print(f"Scenario: {sample['scenario_id']} | Provider: {sample['provider']}")
    print(f"Actions: {sample['actual_actions_taken']} | Required: {sample['n_required_steps']}")
    print()
    for entry in sample["transcript"]:
        role = entry.get("role", "")
        content = entry.get("content", "")
        if role == "thought":
            print(f"Thought: {content[:120]}")
        elif role == "action":
            print(f"Action:  {entry.get('tool', '')}({entry.get('tool_input', '')[:80]})")
        elif role == "observation":
            prefix = "⚠ ERROR Obs" if entry.get("is_error") else "Observation"
            print(f"{prefix}: {content[:120]}")
        elif role == "final_answer":
            print(f"Final Answer: {content[:120]}")
        print()
else:
    print("No completed scenarios yet (run without --dry-run to get real results).")